# 08 — Practical Exception Handling

This is the **8th and final notebook** in the Exception Handling section.

The theory and best practices are already covered. This notebook is intentionally **hands-on**: it combines the concepts you learned into realistic Python programs without introducing another large set of exception-handling concepts.

### Progression

```text
Small Programs
     ↓
Functions
     ↓
Files
     ↓
Data Validation
     ↓
OOP
     ↓
Custom Exceptions
     ↓
Complete Mini Project
```

## 1. Introduction

By now we know:

- `try`
- `except`
- multiple exceptions
- `else`
- `finally`
- `raise`
- custom exceptions
- exception chaining
- best practices

Now we will use them together.

### Learning objectives

By the end of this notebook, you should be able to:

- build robust Python programs
- handle invalid user input
- handle file-related errors
- validate data
- create and use custom exceptions
- handle exceptions at the correct level
- build programs that do not unexpectedly crash
- apply professional exception-handling practices

## 2. Practical Exception-Handling Pattern

A reusable pattern is:

```python
try:
    # Operation that may fail
    pass

except SpecificError:
    # Handle expected problem
    pass

else:
    # Runs when no exception occurs
    pass

finally:
    # Cleanup
    pass
```

We do not necessarily need all four parts every time. The structure depends on the problem.

In [ ]:
try:
    number = int(input("Enter a number: "))
except ValueError:
    print("Invalid number.")

## 3. Project 1 — Safe Calculator

Build a calculator that handles:

- invalid numbers
- division by zero
- invalid operators

In [ ]:
def calculator(a, b, operator):
    if operator == "+":
        return a + b
    elif operator == "-":
        return a - b
    elif operator == "*":
        return a * b
    elif operator == "/":
        return a / b
    else:
        raise ValueError("Invalid operator.")

In [ ]:
try:
    a = float(input("Enter first number: "))
    b = float(input("Enter second number: "))
    operator = input("Enter operator (+, -, *, /): ")

    result = calculator(a, b, operator)

except ValueError as e:
    print("Error:", e)

except ZeroDivisionError:
    print("Cannot divide by zero.")

else:
    print("Result:", result)

The same calculator can later be extended with `%`, `**`, `//`, continuous calculations, and an exit option.

## 4. Project 2 — Safe User Input

Create reusable input functions.

### Integer input

In [ ]:
def get_integer():
    while True:
        try:
            return int(input("Enter an integer: "))
        except ValueError:
            print("Please enter a valid integer.")

In [ ]:
age = get_integer()
print("Age:", age)

### Float input

In [ ]:
def get_float():
    while True:
        try:
            return float(input("Enter a number: "))
        except ValueError:
            print("Please enter a valid number.")

A common practical pattern is:

```text
while
 ↓
try
 ↓
valid input → return
invalid input → except → repeat
```

## 5. Project 3 — Validated User Registration

Requirements:

- name cannot be empty
- age must be an integer
- age must be positive
- email cannot be empty

In [ ]:
def register_user(name, age, email):
    if not name.strip():
        raise ValueError("Name cannot be empty.")

    if age <= 0:
        raise ValueError("Age must be positive.")

    if not email.strip():
        raise ValueError("Email cannot be empty.")

    return {
        "name": name,
        "age": age,
        "email": email
    }

In [ ]:
try:
    name = input("Enter name: ")
    age = int(input("Enter age: "))
    email = input("Enter email: ")

    user = register_user(name, age, email)

except ValueError as e:
    print("Registration failed:", e)

else:
    print("Registration successful.")
    print(user)

## 6. Project 4 — Safe File Reading

Create `read_file(filename)` and handle:

- `FileNotFoundError`
- `PermissionError`
- unexpected errors appropriately

In [ ]:
def read_file(filename):
    try:
        with open(filename, "r") as file:
            return file.read()

    except FileNotFoundError:
        print(f"File '{filename}' does not exist.")

    except PermissionError:
        print("Permission denied.")

In [ ]:
content = read_file("data.txt")

if content is not None:
    print(content)

Using `with open()` is preferable to manually managing the file because the file resource is cleaned up automatically.

## 7. Project 5 — Safe File Writing

Create `write_file(filename, content)`.

In [ ]:
def write_file(filename, content):
    try:
        with open(filename, "w") as file:
            file.write(content)

    except PermissionError:
        print("You do not have permission to write to this file.")

    except OSError as e:
        print("File operation failed:", e)

    else:
        print("File written successfully.")

`OSError` can be useful for broader operating-system file errors when that broader handling is appropriate.

## 8. Project 6 — File Processing

Now combine reading and processing.

In [ ]:
def count_words(filename):
    try:
        with open(filename, "r") as file:
            text = file.read()

        words = text.split()
        return len(words)

    except FileNotFoundError:
        print("File not found.")

    except PermissionError:
        print("Permission denied.")

    except UnicodeDecodeError:
        print("File encoding could not be decoded.")

In [ ]:
count = count_words("sample.txt")

if count is not None:
    print("Word count:", count)

Different operations can fail for different reasons, so specific handlers make the program clearer.

## 9. Project 7 — Safe CSV Processing

This pattern is particularly useful for a **Data Science / Data Analyst** workflow.

In [ ]:
import csv

def read_students(filename):
    try:
        with open(filename, "r", newline="") as file:
            reader = csv.DictReader(file)
            return list(reader)

    except FileNotFoundError:
        print("CSV file not found.")

    except PermissionError:
        print("Permission denied.")

    except csv.Error as e:
        print("CSV processing error:", e)

In [ ]:
students = read_students("students.csv")

if students:
    for student in students:
        print(student)

## 10. Project 8 — CSV Data Validation

Suppose the CSV contains:

```text
name,age,marks
Amit,21,85
Rahul,22,91
Priya,20,abc
```

Converting `abc` to a number raises `ValueError`.

A robust processor can handle an invalid row without losing all valid rows.

In [ ]:
def process_students(filename):
    students = []

    try:
        with open(filename, "r", newline="") as file:
            reader = csv.DictReader(file)

            for row_number, row in enumerate(reader, start=2):
                try:
                    age = int(row["age"])
                    marks = float(row["marks"])

                    students.append({
                        "name": row["name"],
                        "age": age,
                        "marks": marks
                    })

                except ValueError:
                    print(f"Invalid data in row {row_number}")

    except FileNotFoundError:
        print("Student file not found.")

    return students

A useful real-world pattern is:

```text
File-level error
      ↓
Row-level error
      ↓
Continue processing valid rows
```

## 11. Project 9 — Safe Division Function

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None

In [ ]:
result = safe_divide(10, 0)

if result is None:
    print("Division failed.")
else:
    print(result)

### Internal handling vs propagation

A function can handle an error itself:

In [ ]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None

Or it can let the caller handle the exception:

In [ ]:
def divide(a, b):
    return a / b

In [ ]:
try:
    result = divide(10, 0)
except ZeroDivisionError:
    print("Cannot divide by zero.")

The appropriate choice depends on which layer has enough information to meaningfully handle the failure.

## 12. Project 10 — Student Marks System

Combine:

- functions
- lists/dictionaries
- validation
- `raise`
- `try-except`

In [ ]:
students = []

def add_student(name, marks):
    if not name.strip():
        raise ValueError("Student name cannot be empty.")

    if not 0 <= marks <= 100:
        raise ValueError("Marks must be between 0 and 100.")

    students.append({
        "name": name,
        "marks": marks
    })

In [ ]:
try:
    name = input("Enter student name: ")
    marks = float(input("Enter marks: "))

    add_student(name, marks)

except ValueError as e:
    print("Unable to add student:", e)

else:
    print("Student added successfully.")

## 13. Project 11 — Custom Exceptions in Student System

Upgrade the previous project using domain-specific exceptions.

In [ ]:
class InvalidMarksError(Exception):
    pass


class EmptyStudentNameError(Exception):
    pass

In [ ]:
def add_student(name, marks):
    if not name.strip():
        raise EmptyStudentNameError(
            "Student name cannot be empty."
        )

    if not 0 <= marks <= 100:
        raise InvalidMarksError(
            "Marks must be between 0 and 100."
        )

    students.append({
        "name": name,
        "marks": marks
    })

In [ ]:
try:
    add_student("Komal", 105)

except InvalidMarksError as e:
    print("Marks error:", e)

except EmptyStudentNameError as e:
    print("Name error:", e)

This reinforces custom exceptions through application code rather than introducing new exception theory.

## 14. Project 12 — Banking System

Create a small OOP banking system with a custom exception hierarchy.

In [ ]:
class BankingError(Exception):
    pass


class InvalidAmountError(BankingError):
    pass


class InsufficientBalanceError(BankingError):
    pass

In [ ]:
class BankAccount:
    def __init__(self, balance=0):
        self.balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise InvalidAmountError(
                "Deposit amount must be positive."
            )

        self.balance += amount

    def withdraw(self, amount):
        if amount <= 0:
            raise InvalidAmountError(
                "Withdrawal amount must be positive."
            )

        if amount > self.balance:
            raise InsufficientBalanceError(
                "Insufficient balance."
            )

        self.balance -= amount

## 15. Banking System — Exception Handling

In [ ]:
account = BankAccount(5000)

try:
    amount = float(input("Enter withdrawal amount: "))
    account.withdraw(amount)

except ValueError:
    print("Please enter a valid amount.")

except InvalidAmountError as e:
    print("Invalid transaction:", e)

except InsufficientBalanceError as e:
    print("Transaction failed:", e)

else:
    print("Withdrawal successful.")

finally:
    print("Available balance:", account.balance)

This combines almost the entire Exception Handling section.

## 16. Project 13 — Login System

Build a simple login system using a custom authentication exception.

In [ ]:
class AuthenticationError(Exception):
    pass

In [ ]:
def login(username, password):
    correct_username = "admin"
    correct_password = "python123"

    if username != correct_username:
        raise AuthenticationError("Invalid username.")

    if password != correct_password:
        raise AuthenticationError("Invalid password.")

    return True

In [ ]:
try:
    username = input("Username: ")
    password = input("Password: ")

    login(username, password)

except AuthenticationError:
    print("Login failed.")

else:
    print("Login successful.")

### Important security lesson

Never display sensitive credentials such as passwords in error messages or normal output.

## 17. Project 14 — Menu-Driven Application

A common practical pattern is a menu-driven application that continues running after recoverable input errors.

In [ ]:
while True:
    try:
        print("\n===== Student Management System =====")
        print("1. Add Student")
        print("2. View Students")
        print("3. Find Student")
        print("4. Delete Student")
        print("5. Exit")

        choice = int(input("Enter choice: "))

        if choice == 1:
            pass
        elif choice == 2:
            pass
        elif choice == 3:
            pass
        elif choice == 4:
            pass
        elif choice == 5:
            break
        else:
            print("Invalid menu choice.")

    except ValueError:
        print("Please enter a number.")

The practical flow is:

```text
while loop
    ↓
user input
    ↓
validation
    ↓
operation
    ↓
exception handling
    ↓
continue program
```

## 18. Project 15 — Exception Handling with Functions

Create a multi-function application.

In [ ]:
def get_number():
    return int(input("Enter number: "))


def calculate(number):
    if number < 0:
        raise ValueError("Number cannot be negative.")

    return number ** 2

In [ ]:
def main():
    try:
        number = get_number()
        result = calculate(number)

    except ValueError as e:
        print("Error:", e)

    else:
        print("Result:", result)


main()

Exceptions can travel through function calls until a suitable handler handles them.

## 19. Project 16 — Exception Chaining in Data Processing

Create a realistic data-processing situation using a custom exception and exception chaining.

In [ ]:
class DataProcessingError(Exception):
    pass

In [ ]:
def process_age(value):
    try:
        return int(value)

    except ValueError as e:
        raise DataProcessingError(
            f"Invalid age data: {value}"
        ) from e

In [ ]:
try:
    age = process_age("abc")
except DataProcessingError as e:
    print("Data processing failed:", e)

The flow is:

```text
ValueError
    ↓
caught internally
    ↓
DataProcessingError
    ↓
original ValueError preserved as cause
```

## 20. Project 17 — Robust Data Processing Pipeline

This pattern is especially relevant to future Data Science work.

```text
Input File
    ↓
Read Data
    ↓
Validate Data
    ↓
Convert Data Types
    ↓
Process Data
    ↓
Generate Result
```

In [ ]:
def process_record(record):
    try:
        age = int(record["age"])
        marks = float(record["marks"])

    except (ValueError, TypeError) as e:
        raise DataProcessingError(
            "Invalid student record"
        ) from e

    if not 0 <= marks <= 100:
        raise DataProcessingError(
            "Marks must be between 0 and 100"
        )

    return {
        "name": record["name"],
        "age": age,
        "marks": marks
    }

This is a useful bridge between Python exception handling and future Pandas/Data Science projects.

## 21. Handling Partial Failures

Suppose a dataset contains 100 records and 5 of them are invalid.

The entire process does not necessarily need to stop. Valid records can be retained while invalid records are recorded separately.

In [ ]:
valid_records = []
invalid_records = []

for row_number, row in enumerate(rows, start=1):
    try:
        record = process_record(row)
        valid_records.append(record)

    except DataProcessingError as e:
        invalid_records.append({
            "row": row_number,
            "error": str(e)
        })

In [ ]:
print("Valid records:", len(valid_records))
print("Invalid records:", len(invalid_records))

This is a particularly useful pattern for batch data processing.

## 22. Exception Handling with `finally`

A manual cleanup example:

In [ ]:
file = None

try:
    file = open("data.txt", "r")
    content = file.read()

except FileNotFoundError:
    print("File not found.")

finally:
    if file is not None:
        file.close()

    print("Operation completed.")

For files, the preferred approach is normally:

In [ ]:
with open("data.txt", "r") as file:
    content = file.read()

The `with` statement handles file cleanup automatically.

## 23. Project 18 — Robust Expense Manager

This mini project combines data structures, functions, validation, files, and exceptions.

### Features

```text
Expense Manager
│
├── Add Expense
├── View Expenses
├── Calculate Total
├── Save Expenses
├── Load Expenses
└── Exit
```

In [ ]:
class ExpenseError(Exception):
    pass


class InvalidExpenseError(ExpenseError):
    pass

In [ ]:
def add_expense(description, amount):
    if not description.strip():
        raise InvalidExpenseError(
            "Description cannot be empty."
        )

    if amount <= 0:
        raise InvalidExpenseError(
            "Expense amount must be positive."
        )

In [ ]:
try:
    with open("expenses.txt", "a") as file:
        file.write("Example expense\n")
except PermissionError:
    print("Unable to save expense.")

This is a larger practical application that combines the section's major ideas.

## 24. Project 19 — Robust Student Management System

This is the **main project** of the notebook.

### Features

```text
Student Management System
│
├── Add Student
├── View Students
├── Search Student
├── Update Marks
├── Delete Student
├── Calculate Average
├── Save to File
├── Load from File
└── Exit
```

In [ ]:
class StudentError(Exception):
    pass


class StudentNotFoundError(StudentError):
    pass


class InvalidMarksError(StudentError):
    pass


class DuplicateStudentError(StudentError):
    pass

In [ ]:
class StudentManager:
    def __init__(self):
        self.students = []

    def add_student(self, name, marks):
        if not name.strip():
            raise ValueError("Name cannot be empty.")

        if not 0 <= marks <= 100:
            raise InvalidMarksError(
                "Marks must be between 0 and 100."
            )

        for student in self.students:
            if student["name"].lower() == name.lower():
                raise DuplicateStudentError(
                    "Student already exists."
                )

        self.students.append({
            "name": name,
            "marks": marks
        })

    def find_student(self, name):
        for student in self.students:
            if student["name"].lower() == name.lower():
                return student

        raise StudentNotFoundError(
            f"Student '{name}' was not found."
        )

In [ ]:
manager = StudentManager()

try:
    manager.add_student("Komal", 85)
    print(manager.find_student("Komal"))

except StudentError as e:
    print("Student operation failed:", e)

## 25. Main Menu for the Student System

A realistic application boundary can handle menu input and domain-specific student errors.

In [ ]:
def main():
    manager = StudentManager()

    while True:
        print("\n===== Student Management System =====")
        print("1. Add Student")
        print("2. View Students")
        print("3. Search Student")
        print("4. Update Marks")
        print("5. Delete Student")
        print("6. Calculate Average")
        print("7. Exit")

        try:
            choice = int(input("Enter choice: "))

            if choice == 1:
                # Add student
                pass

            elif choice == 2:
                # View students
                pass

            elif choice == 3:
                # Search
                pass

            elif choice == 4:
                # Update
                pass

            elif choice == 5:
                # Delete
                pass

            elif choice == 6:
                # Average
                pass

            elif choice == 7:
                print("Goodbye!")
                break

            else:
                print("Invalid choice.")

        except ValueError:
            print("Please enter a valid menu number.")

        except StudentError as e:
            print("Student operation failed:", e)

This demonstrates handling exceptions at the application boundary.

## 26. Testing Exception Paths

Practical exception handling requires testing failure paths, not only successful cases.

For the Student Management System, consider these cases:

| Case | Input | Expected result |
|---|---|---|
| Valid case | Rahul, 85 | Student added successfully |
| Invalid marks | Rahul, 150 | Marks must be between 0 and 100 |
| Duplicate student | Rahul again | Student already exists |
| Missing student | Search Amit | Student not found |
| Invalid menu | `abc` | Valid menu number message |

> Exception paths are part of the program's behavior and should be tested deliberately.

## 27. Before and After Comparison

Poor code:

```python
try:
    name = input()
    marks = int(input())
    students.append((name, marks))
except:
    pass
```

### Version 1 — Specific exception

```python
try:
    marks = int(input())
except ValueError:
    print("Marks must be an integer.")
```

### Version 2 — Validation

```python
if not 0 <= marks <= 100:
    raise ValueError("Marks must be between 0 and 100.")
```

### Version 3 — Domain-specific exception

```python
class InvalidMarksError(Exception):
    pass
```

### Version 4

Combine these ideas into the Student Management System.

This shows how exception handling evolves from simple error prevention into robust application design.

## 28. Exception Handling Decision Process

When something can go wrong, ask:

```text
Can this situation happen normally?
        │
        ├── Yes
        │    ↓
        │  Use normal validation/control flow
        │
        └── No / exceptional
             ↓
           Could an exception occur?
             ↓
           Catch specific exception?
             ↓
           Can I meaningfully handle it?
             │
             ├── Yes → Handle it
             │
             └── No → Let it propagate
                         ↓
                     Higher-level handler
```

This is a useful mental model for practical programs.

## 29. Final Best-Practice Review

Before considering a practical program complete:

```text
☐ Specific exceptions are caught
☐ No unnecessary bare except
☐ No silent failures
☐ try blocks are focused
☐ Input is validated
☐ Error messages are meaningful
☐ Exceptions are raised where appropriate
☐ Custom exceptions are used only when useful
☐ Original exceptions are preserved when necessary
☐ Exception chaining is used appropriately
☐ Resources are cleaned up
☐ Sensitive information isn't exposed
☐ Programming bugs aren't hidden
☐ Failure cases have been tested
```

## 30. Final Challenge — Complete Application

Build a **Student Management System** from scratch using the concepts from this section.

### Requirements

```text
Student Management System
│
├── Add Student
├── View Students
├── Search Student
├── Update Student
├── Delete Student
├── Calculate Average Marks
├── Save Data to File
├── Load Data from File
└── Exit
```

### Must use

- Classes and objects
- Functions
- Lists/dictionaries
- `try`
- multiple `except`
- `else`
- `finally`
- `raise`
- custom exceptions
- exception hierarchy
- file handling
- input validation
- meaningful error messages

### Must avoid

- bare `except`
- `except Exception` everywhere
- `except: pass`
- huge `try` blocks
- silent failures
- returning from `finally`
- exposing sensitive information

# 31. Exception Handling Section — Final Review

The complete section is:

```text
11_Exception_Handling
│
├── 01_Errors_and_Exceptions.ipynb
├── 02_try_and_except.ipynb
├── 03_Multiple_Exceptions.ipynb
├── 04_else_and_finally.ipynb
├── 05_Raising_Exceptions.ipynb
├── 06_Custom_Exceptions.ipynb
├── 07_Exception_Handling_Best_Practices.ipynb
└── 08_Practical_Exception_Handling.ipynb
```

The learning progression is:

```text
01 → Understand errors
02 → Handle one exception
03 → Handle multiple exceptions
04 → Control success/cleanup paths
05 → Raise exceptions
06 → Create domain-specific exceptions
07 → Write exception handling professionally
08 → Apply everything in real programs
```

## ✅ Exception Handling is now complete

This final notebook is intentionally **more coding-heavy** than the previous notebooks and focuses on applying what you have already learned.

### Next section

```text
12_Modules_and_Packages
```

The next notebook will be:

**`01_Modules_Introduction.ipynb`**